In [ ]:
# ✅ Alapcsomagok frissítése és telepítése
!pip install -q --upgrade pip
!pip install -q timm tqdm

# ✅ Ellenőrzés: GPU elérhető?
import torch
if torch.cuda.is_available():
    print("✅ GPU elérhető:", torch.cuda.get_device_name(0))
else:
    print("❌ Figyelem: Nem találtam GPU-t. Kapcsold be a Runtime > Change runtime type > GPU opciót!")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 69.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 64.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 145.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 36.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 107.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 129.2 MB/s eta 0:00:00
✅ GPU elérhető: NVIDIA A100-SXM4-40GB


✅ 1. Google Drive csatolása + unzip

In [ ]:
from google.colab import drive
import zipfile, os

# Csatolás
drive.mount('/content/drive')

# Útvonalak
diploma_path = "/content/drive/MyDrive/diplomamunka"
unsorted_zip_path = os.path.join(diploma_path, "unsorted_db.zip")
unsorted_dir = "/content/unsorted_db"
sorted_dir = "/content/imagenet_sorted_db"

# Kicsomagolás
with zipfile.ZipFile(unsorted_zip_path, 'r') as zip_ref:
    zip_ref.extractall(unsorted_dir)

print("Kicsomagolva:", unsorted_dir)


Mounted at /content/drive
Kicsomagolva: /content/unsorted_db


✅ 2. Modell betöltése és címketábla előkészítése

In [ ]:
import torch
from torchvision import transforms
from PIL import Image
import timm
import json
from tqdm import tqdm

# Modell betöltése
model = timm.create_model("convnext_base", pretrained=True)
model.eval().cuda()

# ImageNet címkék betöltése
label_url = "https://storage.googleapis.com/download.tensorflow.org/data/imagenet_class_index.json"
import urllib.request
with urllib.request.urlopen(label_url) as url:
    class_idx = json.loads(url.read().decode())
idx_to_label = {int(k): v[1] for k, v in class_idx.items()}

# Transzformáció
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    )
])


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/354M [00:00<?, ?B/s]

Könyvtár korrekció.

In [ ]:
unsorted_dir = os.path.join(unsorted_dir, "unsorted_db")

✅ 3. Első kör: predikció és szortírozás top-1 alapján

In [ ]:
from collections import defaultdict
import shutil

os.makedirs(sorted_dir, exist_ok=True)
class_counts = defaultdict(int)
image_map = dict()
other_dir = os.path.join(sorted_dir, "other")
os.makedirs(other_dir, exist_ok=True)

image_paths = [os.path.join(unsorted_dir, f) for f in os.listdir(unsorted_dir) if f.lower().endswith(".jpg")]

# Első kör: top-1 kategória hozzárendelés
print("Első körös predikció és ideiglenes hozzárendelés...")
for img_path in tqdm(image_paths):
    img = Image.open(img_path).convert("RGB")
    inp = transform(img).unsqueeze(0).cuda()
    with torch.no_grad():
        out = model(inp)
    pred_idx = torch.argmax(out, dim=1).item()
    pred_label = idx_to_label[pred_idx]

    image_map[img_path] = pred_label
    class_counts[pred_label] += 1


Első körös predikció és ideiglenes hozzárendelés...


100%|██████████| 180750/180750 [47:19<00:00, 63.65it/s]


✅ 4. Osztályok kiszűrése <300 példány esetén

In [ ]:
# Kategóriák szűrése
valid_classes = {cls for cls, count in class_counts.items() if count >= 300}
print(f"Maradt {len(valid_classes)} nagy osztály.")

# Újraosztás: első kör fájlmozgatás
for img_path, cls in tqdm(image_map.items()):
    if cls in valid_classes:
        target_dir = os.path.join(sorted_dir, cls)
    else:
        target_dir = other_dir
    os.makedirs(target_dir, exist_ok=True)
    shutil.copy2(img_path, os.path.join(target_dir, os.path.basename(img_path)))


Maradt 114 nagy osztály.


100%|██████████| 180750/180750 [00:31<00:00, 5686.56it/s]


✅ 5. Második kör: other újrabesorolása a megmaradt osztályokba

In [ ]:
other_images = [os.path.join(other_dir, f) for f in os.listdir(other_dir) if f.lower().endswith(".jpg")]

print("Második kör: újraosztályozás a nagy osztályok között...")
for img_path in tqdm(other_images):
    img = Image.open(img_path).convert("RGB")
    inp = transform(img).unsqueeze(0).cuda()
    with torch.no_grad():
        out = model(inp)
    probs = torch.nn.functional.softmax(out, dim=1)

    # Csak valid classokon belül keresünk
    valid_scores = {idx: probs[0, idx].item() for idx, label in idx_to_label.items() if label in valid_classes}
    best_idx = max(valid_scores, key=valid_scores.get)
    best_label = idx_to_label[best_idx]

    # Áthelyezés
    target_dir = os.path.join(sorted_dir, best_label)
    os.makedirs(target_dir, exist_ok=True)
    shutil.move(img_path, os.path.join(target_dir, os.path.basename(img_path)))


Második kör: újraosztályozás a nagy osztályok között...


100%|██████████| 44027/44027 [13:16<00:00, 55.25it/s]


✅ 6. Tömörítés és visszamásolás a Drive-ba

In [ ]:
print("Tömörítés folyamatban...")
shutil.make_archive("imagenet_sorted_db", 'zip', sorted_dir)

print("Visszamásolás a Drive-ba...")
shutil.move("/content/imagenet_sorted_db.zip", os.path.join(diploma_path, "imagenet_sorted_db.zip"))

print("✅ Kész!")


Tömörítés folyamatban...
Visszamásolás a Drive-ba...
✅ Kész!
